# Single Ticker CIO Demo

This notebook runs the three current agents for one ticker, converts their outputs into CIO packets, then asks the CIO layer for a final decision and a single-ticker event-window eval.


In [12]:
import os
import sys
import json
import warnings
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from dotenv import load_dotenv

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Pandas requires version .* of 'numexpr'.*")

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

loaded_env = load_dotenv(REPO_ROOT / ".env", override=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Loaded .env: {loaded_env}")
print(f"VERTEX_PROJECT loaded: {bool(os.getenv('VERTEX_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT'))}")
print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"NEWSAPI_KEY loaded: {bool(os.getenv('NEWSAPI_KEY'))}")
print(f"FINNHUB_API_KEY loaded: {bool(os.getenv('FINNHUB_API_KEY'))}")


Repo root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory
Loaded .env: True
VERTEX_PROJECT loaded: True
OPENAI_API_KEY loaded: False
NEWSAPI_KEY loaded: True
FINNHUB_API_KEY loaded: True


In [13]:
import importlib

import openclam.agents.news_macro.news_macro_agent as news_macro_agent
import openclam.agents.market_technical.market_technical_agent as market_technical_agent
import openclam.agents.fundamental.fundamental_agent as fundamental_agent
import openclam.agents.cio.cio_agent as cio_agent

news_macro_agent = importlib.reload(news_macro_agent)
market_technical_agent = importlib.reload(market_technical_agent)
fundamental_agent = importlib.reload(fundamental_agent)
cio_agent = importlib.reload(cio_agent)

print(f"Loaded news_macro_agent from: {news_macro_agent.__file__}")
print(f"Loaded market_technical_agent from: {market_technical_agent.__file__}")
print(f"Loaded fundamental_agent from: {fundamental_agent.__file__}")
print(f"Loaded cio_agent from: {cio_agent.__file__}")


Loaded news_macro_agent from: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/agents/news_macro/news_macro_agent.py
Loaded market_technical_agent from: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/agents/market_technical/market_technical_agent.py
Loaded fundamental_agent from: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/agents/fundamental/fundamental_agent.py
Loaded cio_agent from: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/agents/cio/cio_agent.py


## Configure One Ticker

Use an earnings date when you want post-earnings event-window evaluation. For latest analysis, switch `NEWS_MODE` to `latest` and keep `EVENT_DATE` as today or the event anchor you care about.


In [17]:
TICKER = "PLTR"
COMPANY = "Palantir"
EVENT_DATE = "2026-02-03"
NEWS_MODE = "event_window"
LOOKBACK_DAYS = 14
MAX_NEWS = 10

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30
NEUTRAL_BAND = 0.02

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
# VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-pro")
LLM_PROVIDER = "vertex" if VERTEX_PROJECT else "openai"

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
FUNDAMENTAL_MODEL = os.getenv("FUNDAMENTAL_MODEL", OPENAI_MODEL)
MARKET_MODEL = os.getenv("MARKET_MODEL", OPENAI_MODEL)
NEWS_MODEL = os.getenv("NEWS_MODEL", "gpt-5.4-nano")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

# Some GPT-5 family models only support their default temperature.
def supported_temperature(model_name: str, preferred: float = 0.0) -> float:
    model_name = (model_name or "").lower()
    return 1.0 if model_name.startswith("gpt-5") else preferred

FUNDAMENTAL_TEMPERATURE = supported_temperature(FUNDAMENTAL_MODEL, preferred=0.0)
MARKET_TEMPERATURE = supported_temperature(MARKET_MODEL, preferred=0.2)

TRANSCRIPT_YEAR = 2025
TRANSCRIPT_QUARTER = 4

print(f"LLM_PROVIDER: {LLM_PROVIDER}")
print(f"VERTEX_MODEL: {VERTEX_MODEL}")


LLM_PROVIDER: vertex
VERTEX_MODEL: gemini-2.5-pro


## Vertex AI Adapters

These adapters let the existing Fundamental and Market/Technical agents keep their original interfaces while using Vertex Gemini underneath.


In [18]:
class VertexTextGenerator:
    def __init__(self, project: str, location: str = "us-central1", model: str = "gemini-2.5-flash"):
        self.project = project
        self.location = location
        self.model = model
        self.backend = None
        self.client = None
        self.vertex_model = None

        try:
            from google import genai
            self.client = genai.Client(vertexai=True, project=project, location=location)
            self.backend = "google_genai"
        except Exception:
            import vertexai
            from vertexai.generative_models import GenerativeModel
            vertexai.init(project=project, location=location)
            self.vertex_model = GenerativeModel(model)
            self.backend = "vertexai"

    def generate(self, messages, temperature: float = 1.0, json_mode: bool = True) -> str:
        prompt = "".join(
            f"{message.get('role', 'user').upper()}: {message.get('content', '')}"
            for message in messages
        )
        if self.backend == "google_genai":
            from google.genai import types
            config = types.GenerateContentConfig(
                temperature=temperature,
                max_output_tokens=8192,
                response_mime_type="application/json" if json_mode else None,
            )
            response = self.client.models.generate_content(
                model=self.model,
                contents=prompt,
                config=config,
            )
            return response.text or "{}"

        from vertexai.generative_models import GenerationConfig
        response = self.vertex_model.generate_content(
            prompt,
            generation_config=GenerationConfig(
                temperature=temperature,
                max_output_tokens=8192,
                response_mime_type="application/json" if json_mode else None,
            ),
        )
        return response.text or "{}"


class VertexOpenAICompatibleClient:
    def __init__(self, generator: VertexTextGenerator):
        self.generator = generator
        self.chat = SimpleNamespace(completions=self)

    def create(self, model=None, temperature=1.0, messages=None, **kwargs):
        text = self.generator.generate(messages or [], temperature=temperature, json_mode=True)
        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(content=text))]
        )


class VertexLangChainCompatibleLLM:
    def __init__(self, generator: VertexTextGenerator, temperature: float = 1.0):
        self.generator = generator
        self.temperature = temperature

    def invoke(self, messages):
        text = self.generator.generate(messages or [], temperature=self.temperature, json_mode=True)
        return SimpleNamespace(content=text)


def create_preferred_vertex_generator():
    if not VERTEX_PROJECT:
        return None
    return VertexTextGenerator(
        project=VERTEX_PROJECT,
        location=VERTEX_LOCATION,
        model=VERTEX_MODEL,
    )

vertex_generator = create_preferred_vertex_generator()
print(f"Vertex generator active: {bool(vertex_generator)}")


def vertex_error_hint(exc) -> str:
    text = repr(exc)
    if "DefaultCredentialsError" in text or "default credentials" in text:
        return (
            "Vertex ADC is missing. Run `gcloud auth application-default login` locally, "
            "or let this notebook fall back to OpenAI if OPENAI_API_KEY is set."
        )
    return text

def has_openai_fallback() -> bool:
    return bool(os.getenv("OPENAI_API_KEY"))


Vertex generator active: True


## Build Single-Ticker Eval Frame

This gives CIO a realized benchmark comparison for the selected earnings event. The eval target is abnormal return versus QQQ, not raw stock return.


In [19]:
earnings_df = pd.DataFrame([
    {"ticker": TICKER, "company": COMPANY, "earnings_date": EVENT_DATE}
])

summary_df, paths_df = news_macro_agent.build_earnings_price_eval(
    earnings_df=earnings_df,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    benchmarks=("SPY", "QQQ"),
)

news_macro_agent.format_return_columns(summary_df)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,abnormal_vs_spy,spy_post_30d_return,abnormal_30d_vs_spy,qqq_post_7d_return,abnormal_vs_qqq,qqq_post_30d_return,abnormal_30d_vs_qqq,realized_direction_vs_qqq,realized_30d_direction_vs_qqq,long_horizon_trading_days
0,PLTR,Palantir,2026-02-03,-6.91%,-11.62%,-13.92%,-18.21%,-3.24%,-23.86%,-9.92%,...,-17.01%,-4.08%,0.84%,-2.58%,-15.63%,-3.51%,0.27%,down,up,30


## Run Three Agents

Each block writes either an output object or an error message into `agent_errors`. CIO will consume every successful packet.


In [25]:
agent_outputs = {}
agent_errors = {}

# 1. News & Macro Agent: Vertex first, then OpenAI/Gemini/heuristic through provider='auto' if no Vertex project.
try:
    news_context = news_macro_agent.collect_context(
        ticker=TICKER,
        company=COMPANY,
        event_date=EVENT_DATE,
        lookback_days=LOOKBACK_DAYS,
        max_news=MAX_NEWS,
        news_mode=NEWS_MODE,
        news_sources=["finnhub", "newsapi", "yfinance"],
        use_sample_if_empty=False,
        news_end_offset_days=0,
    )
    news_report = news_macro_agent.generate_report(
        news_context,
        # provider="vertex" if VERTEX_PROJECT else "auto",
        provider="auto",
        model=NEWS_MODEL,
        gemini_model=VERTEX_MODEL if VERTEX_PROJECT else GEMINI_MODEL,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
    )
    agent_outputs["news_macro"] = news_report
except Exception as exc:
    agent_errors["news_macro"] = repr(exc)

# 2. Market & Technical Agent: Vertex first; OpenAI fallback if Vertex credentials are unavailable.
try:
    try:
        if not vertex_generator:
            raise RuntimeError("Vertex generator is not configured.")
        market_llm = VertexLangChainCompatibleLLM(vertex_generator, temperature=1.0)
        market_query = f"{COMPANY} ({TICKER}) technical analysis as of {EVENT_DATE.replace('-', '/')}"
        market_report = market_technical_agent.run_market_analysis(
            market_query,
            llm=market_llm,
            ticker=TICKER,
        )
        agent_outputs["market_technical"] = market_report
    except Exception as vertex_exc:
        agent_errors["market_technical_vertex"] = vertex_error_hint(vertex_exc)
        if not has_openai_fallback():
            raise
        market_llm = market_technical_agent.create_market_llm(
            api_key=os.getenv("OPENAI_API_KEY"),
            model=MARKET_MODEL,
            temperature=MARKET_TEMPERATURE,
        )
        market_query = f"{COMPANY} ({TICKER}) technical analysis as of {EVENT_DATE.replace('-', '/')}"
        market_report = market_technical_agent.run_market_analysis(
            market_query,
            llm=market_llm,
            ticker=TICKER,
        )
        agent_outputs["market_technical"] = market_report
        agent_errors["market_technical_fallback"] = "Used OpenAI fallback after Vertex failure."
except Exception as exc:
    agent_errors["market_technical"] = repr(exc)

# 3. Fundamental Agent: Vertex first; OpenAI fallback if Vertex credentials are unavailable.
try:
    try:
        if not vertex_generator:
            raise RuntimeError("Vertex generator is not configured.")
        fundamental_client = VertexOpenAICompatibleClient(vertex_generator)
        fundamental_result = fundamental_agent.run_fundamental_analysis(
            ticker=TICKER,
            transcript_year=TRANSCRIPT_YEAR,
            transcript_quarter=TRANSCRIPT_QUARTER,
            require_transcript=False,
            model_name=VERTEX_MODEL,
            temperature=1.0,
            client=fundamental_client,
        )
        agent_outputs["fundamental"] = fundamental_result
    except Exception as vertex_exc:
        agent_errors["fundamental_vertex"] = vertex_error_hint(vertex_exc)
        if not has_openai_fallback():
            raise
        fundamental_result = fundamental_agent.run_fundamental_analysis(
            ticker=TICKER,
            transcript_year=TRANSCRIPT_YEAR,
            transcript_quarter=TRANSCRIPT_QUARTER,
            require_transcript=False,
            model_name=FUNDAMENTAL_MODEL,
            temperature=FUNDAMENTAL_TEMPERATURE,
        )
        agent_outputs["fundamental"] = fundamental_result
        agent_errors["fundamental_fallback"] = "Used OpenAI fallback after Vertex failure."
except Exception as exc:
    agent_errors["fundamental"] = repr(exc)

print("Successful agents:", list(agent_outputs.keys()))
print("Agent errors:", agent_errors)


Successful agents: ['news_macro', 'market_technical', 'fundamental']
Agent errors: {}


In [26]:
TICKER = "PLTR"
COMPANY = "Palantir"
EVENT_DATE = "2026-02-03"
NEWS_MODE = "event_window"
LOOKBACK_DAYS = 14
MAX_NEWS = 10

In [32]:
# Quick human-readable outputs
if "news_macro" in agent_outputs:
    print("NEWS/MACRO")
    print(agent_outputs["news_macro"].short_term_stance, agent_outputs["news_macro"].long_term_stance, agent_outputs["news_macro"].confidence_score)
    print(agent_outputs["news_macro"].stance_rationale[:500])

if "market_technical" in agent_outputs:
    print("MARKET/TECHNICAL")
    print(agent_outputs["market_technical"].get("short_term_stance"), agent_outputs["market_technical"].get("long_term_stance"), agent_outputs["market_technical"].get("confidence"))
    print(agent_outputs["market_technical"].get("core_insight") or agent_outputs["market_technical"].get("summary"))

if "fundamental" in agent_outputs:
    print("FUNDAMENTAL")
    print(agent_outputs["fundamental"].stance, agent_outputs["fundamental"].confidence)
    print(agent_outputs["fundamental"].core_judgment[:500])


NEWS/MACRO
Bullish Bullish 0.9
Short-term: The blowout earnings beat, significant guidance raise, and massive margin expansion, combined with a 12% pre-earnings pullback, create a perfect setup for a sharp upward re-rating versus the market. Long-term: Palantir is demonstrating a successful transition to a scalable, high-margin commercial AI platform, justifying a premium valuation and suggesting sustained fundamental outperformance as it takes share from less effective solutions.
MARKET/TECHNICAL
bearish bearish 0.9
The market's overwhelmingly negative reaction to earnings, confirmed by a -9.87% abnormal return vs. QQQ and massive volume, has decisively broken the stock's long-term technical structure (below the MA200), signaling a probable bearish trend continuation.
FUNDAMENTAL
Bullish 0.5
The company reported an exceptionally strong quarter with significant quarter-over-quarter acceleration in revenue, profitability, and cash flow. The balance sheet is robust with a substantial net

## Convert Agent Outputs to CIO Packets


In [29]:
cio_packets = []

if "news_macro" in agent_outputs:
    cio_packets.append(news_macro_agent.to_cio_agent_input(agent_outputs["news_macro"]))

if "market_technical" in agent_outputs:
    cio_packets.append(cio_agent.to_cio_packet_from_market(agent_outputs["market_technical"]))

if "fundamental" in agent_outputs:
    cio_packets.append(cio_agent.to_cio_packet_from_fundamental(agent_outputs["fundamental"], company=COMPANY))

if not cio_packets:
    raise RuntimeError(f"No CIO packets were created. Agent errors: {agent_errors}")

pd.DataFrame([
    {
        "agent_name": packet["agent_name"],
        "short_term_stance": packet["short_term_stance"],
        "long_term_stance": packet["long_term_stance"],
        "confidence": packet["confidence"],
        "rationale": packet["stance_rationale"][:220],
    }
    for packet in cio_packets
])


,agent_name,short_term_stance,long_term_stance,confidence,rationale
0,news_macro,Bullish,Bullish,0.9,"Short-term: The blowout earnings beat, signifi..."
1,market_technical,Bearish,Bearish,0.9,The market's overwhelmingly negative reaction ...
2,fundamental,Neutral,Bullish,0.5,The company reported an exceptionally strong q...


## Run CIO Decision

Set `use_llm_debate=True` and `use_llm_decision=True` when you want the CIO layer itself to use OpenAI for debate and final synthesis. With both set to `False`, CIO uses the transparent deterministic fallback.


In [30]:
cio_result = cio_agent.run_cio_workflow(
    cio_packets,
    use_llm_debate=False,
    use_llm_decision=False,
    llm_provider=LLM_PROVIDER,
    debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    vertex_project=VERTEX_PROJECT,
    vertex_location=VERTEX_LOCATION,
)

cio_result["final_decision"]


{'ticker': 'PLTR',
 'company': 'Palantir',
 'final_short_term_stance': 'Neutral',
 'final_long_term_stance': 'Bullish',
 'investment_action': 'Watch / No Trade',
 'position_size_hint': 'No new position',
 'confidence': 0.402,
 'core_thesis': 'CIO view for PLTR: short-term neutral, long-term bullish, based on weighted cross-agent evidence.',
 'why_now': 'Decision is based on the current event-window evidence packet supplied by the agents.',
 'key_supporting_evidence': ['Positive sympathetic move.',
  'Uncertain. Could be negative due to competitive concerns or positive if seen as a sign of a larger data analytics market.',
  'Slightly positive.',
  'Potentially negative.',
  "Because Palantir's AI platform growth is driving a surge in high-density compute deployments, the demand for specialized power and liquid cooling solutions inside data centers will accelerate. This directly increases the order book and pricing power for Vertiv, a pure-play leader in this critical infrastructure.",


In [31]:
cio_result["synthesis"]

{'ticker': 'PLTR',
 'company': 'Palantir',
 'agent_view_summary': [{'agent_name': 'news_macro',
   'short_term_stance': 'Bullish',
   'long_term_stance': 'Bullish',
   'confidence': 0.9,
   'rationale': 'Short-term: The blowout earnings beat, significant guidance raise, and massive margin expansion, combined with a 12% pre-earnings pullback, create a perfect setup for a sharp upward re-rating versus the market. Long-term: Palantir is demonstrating a successful transition to a scalable, high-margin commercial AI platform, justifying a premium valuation and suggesting sustained fundamental outperformance as it takes share from less effective solutions.'},
  {'agent_name': 'market_technical',
   'short_term_stance': 'Bearish',
   'long_term_stance': 'Bearish',
   'confidence': 0.9,
   'rationale': "The market's overwhelmingly negative reaction to earnings, confirmed by a -9.87% abnormal return vs. QQQ and massive volume, has decisively broken the stock's long-term technical structure (bel

In [ ]:
cio_result["debate"]

{'triggered': False,
 'trigger': {'debate_required': False,
  'conflict_level': 'low',
  'debate_reason': ['No material cross-agent disagreement.']},
 'synthesis': {'ticker': 'NVDA',
  'company': 'Nvidia',
  'agent_view_summary': [{'agent_name': 'news_macro',
    'short_term_stance': 'Bullish',
    'long_term_stance': 'Bullish',
    'confidence': 0.8,
    'rationale': "The short-term stance is Bullish due to a definitive earnings and guidance beat that exceeded high investor expectations, triggering positive capital flows. The long-term stance remains Bullish because the guidance implies accelerating growth over the next quarter, and demand for NVDA's current and next-generation products (Vera Rubin) appears to overwhelm the more distant threats from competition and geopolitics for the next 1-3 quarters."},
   {'agent_name': 'market_technical',
    'short_term_stance': 'Neutral',
    'long_term_stance': 'Bullish',
    'confidence': 0.85,
    'rationale': 'The dominant technical feature

## Single-Ticker CIO Eval

This compares CIO's final short-term stance against 7-trading-day abnormal return vs QQQ, and CIO's long-term stance against 30-trading-day abnormal return vs QQQ.


In [33]:
packets_by_ticker = {TICKER: cio_packets}

cio_eval, cio_results = cio_agent.run_cio_eval(
    summary_df,
    packets_by_ticker,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
    use_llm_debate=False,
    use_llm_decision=False,
    llm_provider=LLM_PROVIDER,
    debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    vertex_project=VERTEX_PROJECT,
    vertex_location=VERTEX_LOCATION,
)

news_macro_agent.format_return_columns(cio_eval)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_routing_bucket,cio_routing_rationale,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,PLTR,Palantir,2026-02-03,-6.91%,-11.62%,-13.92%,-18.21%,-3.24%,-23.86%,-9.92%,...,0.402,True,high,default,Default CIO weights are used because no bucket...,Short score -0.2113 and long score 0.4737 afte...,False,neutral missed: abnormal return moved outside ...,True,matched


In [34]:
cio_agent.evaluate_cio_decisions(cio_eval)


{'cases': 1,
 'cio_ready_cases': 1,
 'debate_trigger_rate': 1.0,
 'cio_short_direction_match_evaluable': 1,
 'cio_short_direction_match_matched': 0,
 'cio_short_direction_match_accuracy': 0.0,
 'cio_long_direction_match_evaluable': 1,
 'cio_long_direction_match_matched': 1,
 'cio_long_direction_match_accuracy': 1.0}

## Magnificent 7 CIO Eval

This runs the same three-agent-to-CIO pipeline for all Magnificent 7 Q4 2025 earnings anchors. It can take several minutes and uses multiple model calls.


In [14]:
def run_three_agent_packets_for_ticker(ticker: str, company: str, event_date: str):
    outputs = {}
    errors = {}

    try:
        context = news_macro_agent.collect_context(
            ticker=ticker,
            company=company,
            event_date=event_date,
            lookback_days=LOOKBACK_DAYS,
            max_news=MAX_NEWS,
            news_mode="event_window",
            news_sources=["finnhub", "newsapi", "yfinance"],
            use_sample_if_empty=False,
            news_end_offset_days=0,
        )
        outputs["news_macro"] = news_macro_agent.generate_report(
            context,
            provider="vertex" if VERTEX_PROJECT else "auto",
            model=NEWS_MODEL,
            gemini_model=VERTEX_MODEL if VERTEX_PROJECT else GEMINI_MODEL,
            vertex_project=VERTEX_PROJECT,
            vertex_location=VERTEX_LOCATION,
        )
    except Exception as exc:
        errors["news_macro"] = repr(exc)

    try:
        try:
            if not vertex_generator:
                raise RuntimeError("Vertex generator is not configured.")
            market_llm = VertexLangChainCompatibleLLM(vertex_generator, temperature=1.0)
            query = f"{company} ({ticker}) technical analysis as of {event_date.replace('-', '/')}"
            outputs["market_technical"] = market_technical_agent.run_market_analysis(query, llm=market_llm, ticker=ticker)
        except Exception as vertex_exc:
            errors["market_technical_vertex"] = vertex_error_hint(vertex_exc)
            if not has_openai_fallback():
                raise
            market_llm = market_technical_agent.create_market_llm(
                api_key=os.getenv("OPENAI_API_KEY"),
                model=MARKET_MODEL,
                temperature=MARKET_TEMPERATURE,
            )
            query = f"{company} ({ticker}) technical analysis as of {event_date.replace('-', '/')}"
            outputs["market_technical"] = market_technical_agent.run_market_analysis(query, llm=market_llm, ticker=ticker)
            errors["market_technical_fallback"] = "Used OpenAI fallback after Vertex failure."
    except Exception as exc:
        errors["market_technical"] = repr(exc)

    try:
        try:
            if not vertex_generator:
                raise RuntimeError("Vertex generator is not configured.")
            fundamental_client = VertexOpenAICompatibleClient(vertex_generator)
            outputs["fundamental"] = fundamental_agent.run_fundamental_analysis(
                ticker=ticker,
                transcript_year=TRANSCRIPT_YEAR,
                transcript_quarter=TRANSCRIPT_QUARTER,
                require_transcript=False,
                model_name=VERTEX_MODEL,
                temperature=1.0,
                client=fundamental_client,
            )
        except Exception as vertex_exc:
            errors["fundamental_vertex"] = vertex_error_hint(vertex_exc)
            if not has_openai_fallback():
                raise
            outputs["fundamental"] = fundamental_agent.run_fundamental_analysis(
                ticker=ticker,
                transcript_year=TRANSCRIPT_YEAR,
                transcript_quarter=TRANSCRIPT_QUARTER,
                require_transcript=False,
                model_name=FUNDAMENTAL_MODEL,
                temperature=FUNDAMENTAL_TEMPERATURE,
            )
            errors["fundamental_fallback"] = "Used OpenAI fallback after Vertex failure."
    except Exception as exc:
        errors["fundamental"] = repr(exc)

    packets = []
    if "news_macro" in outputs:
        packets.append(news_macro_agent.to_cio_agent_input(outputs["news_macro"]))
    if "market_technical" in outputs:
        packets.append(cio_agent.to_cio_packet_from_market(outputs["market_technical"]))
    if "fundamental" in outputs:
        packets.append(cio_agent.to_cio_packet_from_fundamental(outputs["fundamental"], company=company))

    return packets, outputs, errors


mag7_earnings_df = news_macro_agent.mag7_q4_2025_earnings_df()
mag7_summary_df, mag7_paths_df = news_macro_agent.build_earnings_price_eval(
    earnings_df=mag7_earnings_df,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    benchmarks=("SPY", "QQQ"),
)

mag7_packets_by_ticker = {}
mag7_agent_outputs = {}
mag7_agent_errors = {}
for row in mag7_earnings_df.itertuples(index=False):
    print(f"Running agents for {row.ticker}...")
    packets, outputs, errors = run_three_agent_packets_for_ticker(row.ticker, row.company, row.earnings_date)
    mag7_packets_by_ticker[row.ticker] = packets
    mag7_agent_outputs[row.ticker] = outputs
    mag7_agent_errors[row.ticker] = errors

mag7_agent_errors


Running agents for TSLA...
Running agents for META...
Running agents for AAPL...
Running agents for MSFT...
Running agents for GOOGL...
Running agents for AMZN...
Running agents for NVDA...


{'TSLA': {},
 'META': {},
 'AAPL': {},
 'MSFT': {},
 'GOOGL': {},
 'AMZN': {},
 'NVDA': {}}

In [19]:
mag7_cio_eval, mag7_cio_results = cio_agent.run_cio_eval(
    mag7_summary_df,
    mag7_packets_by_ticker,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
    use_llm_debate=True,
    use_llm_decision=True,
    llm_provider=LLM_PROVIDER,
    debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
    vertex_project=VERTEX_PROJECT,
    vertex_location=VERTEX_LOCATION,
)

news_macro_agent.format_return_columns(mag7_cio_eval)


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_long_term_stance,cio_action,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,Bearish,Short / Avoid,0.85,True,high,I am confirming the deterministic guardrail's ...,True,matched,True,matched
1,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,Neutral,Reduce / Avoid,0.80,True,high,I am overriding the deterministic guardrail's ...,False,missed,True,neutral matched: abnormal return stayed inside...
2,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,Neutral,Reduce / Trim,0.80,True,high,This is a classic case of a forward-looking fu...,False,missed,False,neutral missed: abnormal return moved outside ...
3,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,Neutral,Watch / No Trade,0.70,False,low,The final decision aligns with the determinist...,True,matched,False,neutral missed: abnormal return moved outside ...
4,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,Neutral,Reduce / Sell,0.85,True,high,This is a high-conviction override of the guar...,True,matched,False,neutral missed: abnormal return moved outside ...
5,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,Bullish,Hold / Do Not Add,0.80,True,high,This decision constitutes an override of the d...,True,matched,False,missed
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,Bullish,Long,0.80,False,low,The final decision is to adopt a Bullish/Bulli...,False,missed,False,missed


In [20]:
cio_agent.evaluate_cio_decisions(mag7_cio_eval)


{'cases': 7,
 'cio_ready_cases': 7,
 'debate_trigger_rate': 0.7142857142857143,
 'cio_short_direction_match_evaluable': 7,
 'cio_short_direction_match_matched': 4,
 'cio_short_direction_match_accuracy': 0.5714285714285714,
 'cio_long_direction_match_evaluable': 7,
 'cio_long_direction_match_matched': 2,
 'cio_long_direction_match_accuracy': 0.2857142857142857}

## Check for agent debates

In [37]:
from pprint import pprint
import pandas as pd

TICKER_TO_INSPECT = "TSLA"  # You can change to "META", "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA"

workflow = mag7_cio_results[TICKER_TO_INSPECT]
synthesis = workflow["synthesis"]
debate = workflow["debate"]
decision = workflow["final_decision"]

print("=" * 90)
print(f"CIO Debate Inspection: {TICKER_TO_INSPECT}")
print("=" * 90)

print("\n[1] Debate Trigger")
pprint(debate["trigger"])

print("\n[2] Areas of Agreement")
pprint(synthesis.get("areas_of_agreement", []))

print("\n[3] Areas of Disagreement")
pprint(synthesis.get("areas_of_disagreement", []))

print("\n[4] Original Agent Views")
original_views = pd.DataFrame(synthesis["agent_view_summary"])
display(original_views)

print("\n[5] Debate Responses")
if debate.get("triggered"):
    debate_responses = pd.DataFrame(debate.get("debate_responses", []))
    display(debate_responses)
else:
    print("No debate was triggered for this ticker.")

print("\n[6] Revised Agent Packets After Debate")
revised_rows = []
for packet in debate.get("revised_packets", []):
    revised_rows.append({
        "agent_name": packet.get("agent_name"),
        "revised_short_term_stance": packet.get("short_term_stance"),
        "revised_long_term_stance": packet.get("long_term_stance"),
        "revised_confidence": packet.get("confidence"),
        "rationale": packet.get("stance_rationale"),
    })

display(pd.DataFrame(revised_rows))

print("\n[7] Final CIO Decision")
pprint({
    "final_short_term_stance": decision.get("final_short_term_stance"),
    "final_long_term_stance": decision.get("final_long_term_stance"),
    "investment_action": decision.get("investment_action"),
    "position_size_hint": decision.get("position_size_hint"),
    "confidence": decision.get("confidence"),
    "reason_for_final_decision": decision.get("reason_for_final_decision"),
    "scores": decision.get("scores"),
})

print("\n[8] Agent Votes")
print("\nShort-term votes:")
display(pd.DataFrame(decision["agent_votes"]["short"]))

print("\nLong-term votes:")
display(pd.DataFrame(decision["agent_votes"]["long"]))


NameError: name 'mag7_cio_results' is not defined

In [22]:
for ticker, workflow in mag7_cio_results.items():
    debate = workflow["debate"]
    decision = workflow["final_decision"]

    if not debate.get("triggered"):
        continue

    print("\n" + "=" * 100)
    print(f"{ticker} | Conflict level: {debate['trigger']['conflict_level']}")
    print("=" * 100)

    print("\nDebate reason:")
    pprint(debate["trigger"]["debate_reason"])

    print("\nOriginal views:")
    display(pd.DataFrame(workflow["synthesis"]["agent_view_summary"]))

    print("\nDebate responses:")
    display(pd.DataFrame(debate.get("debate_responses", [])))

    print("\nRevised packets:")
    display(pd.DataFrame([
        {
            "agent_name": p.get("agent_name"),
            "short_term_stance": p.get("short_term_stance"),
            "long_term_stance": p.get("long_term_stance"),
            "confidence": p.get("confidence"),
        }
        for p in debate.get("revised_packets", [])
    ]))

    print("\nFinal CIO:")
    pprint({
        "short": decision.get("final_short_term_stance"),
        "long": decision.get("final_long_term_stance"),
        "action": decision.get("investment_action"),
        "confidence": decision.get("confidence"),
        "reason": decision.get("reason_for_final_decision"),
    })



TSLA | Conflict level: high

Debate reason:
['Short-term views contain both bullish and bearish calls.',
 'High-confidence short-term agents disagree.']

Original views:


,agent_name,short_term_stance,long_term_stance,confidence,rationale
0,news_macro,Bearish,Bearish,0.8,Short-term: The top-line revenue miss and year...
1,market_technical,Bullish,Neutral,0.7,The key technical dynamic is a conflict betwee...
2,fundamental,Neutral,Neutral,0.3,The company reported a significant quarter-ove...



Debate responses:


,response_type,agreement,disagreement,revised_short_term_stance,revised_long_term_stance,revised_confidence,stance_revision_rationale,evidence_needed
0,CIO Committee Revision,{'market_technical': 'I agree with the market_...,{'market_technical': 'I disagree with the mark...,Neutral,Bearish,0.6,I am revising my short-term stance from Bearis...,[Clarification on the discrepancy between the ...
1,revision,I agree with the core evidence presented by th...,I now disagree with the weight I previously pl...,Bearish,Bearish,0.9,NaN,The primary evidence needed now is the market'...
2,revision,I agree with the news_macro agent's primary ev...,I assign low weight to the market_technical ag...,Bearish,Bearish,0.8,NaN,[Official management guidance for forward-look...



Revised packets:


...


Debate responses:


,response_type,agreement,disagreement,revised_short_term_stance,revised_long_term_stance,revised_confidence,evidence_needed
0,CIO_committee_revision,{'description': 'I agree with specific data po...,{'description': 'I disagree with the overall s...,"{'stance': 'Bearish', 'change': 'Maintained', ...","{'stance': 'Bullish', 'change': 'Maintained', ...","{'confidence': 0.8, 'change': 'Increased', 'ra...",[The earnings call transcript to analyze manag...
1,revision,{'statement': 'I agree with the core evidence ...,{'statement': 'I must revise my original short...,Neutral,Bullish,0.65,[Price action confirmation: I need to observe ...
2,revision,I agree with the core evidence provided by the...,I disagree with the market_technical agent's '...,Bearish,Bullish,0.75,[Official management guidance on the quarterly...



Revised packets:


,agent_name,short_term_stance,long_term_stance,confidence
0,news_macro,Neutral,Neutral,0.70
1,market_technical,Neutral,Bullish,0.65
2,fundamental,Bearish,Bullish,0.75



Final CIO:
{'action': 'Hold / Do Not Add',
 'confidence': 0.8,
 'long': 'Bullish',
 'reason': 'This decision constitutes an override of the deterministic '
           "guardrail's 'Neutral' short-term stance. The post-debate synthesis "
           "provides overwhelming and coherent evidence for a 'Bearish' "
           'short-term view. The news of the $200B capex plan, identified by '
           'the news_macro agent, was the critical piece of information that '
           "unified the committee's view. It explained the shocking -$18.2B "
           'free cash flow figure from the fundamental agent and acted as the '
           'catalyst that broke the bullish trend identified by the technical '
           'agent. The unanimous post-debate consensus is that the market will '
           'punish the stock in the near term for this cash flow compression, '
           'even while the long-term strategic rationale for the investment is '
           'sound.',
 'short': 'Bearish'}


In [23]:
pd.DataFrame(mag7_cio_results["TSLA"]["debate"]["debate_responses"])


,response_type,agreement,disagreement,revised_short_term_stance,revised_long_term_stance,revised_confidence,stance_revision_rationale,evidence_needed
0,CIO Committee Revision,{'market_technical': 'I agree with the market_...,{'market_technical': 'I disagree with the mark...,Neutral,Bearish,0.6,I am revising my short-term stance from Bearis...,[Clarification on the discrepancy between the ...
1,revision,I agree with the core evidence presented by th...,I now disagree with the weight I previously pl...,Bearish,Bearish,0.9,NaN,The primary evidence needed now is the market'...
2,revision,I agree with the news_macro agent's primary ev...,I assign low weight to the market_technical ag...,Bearish,Bearish,0.8,NaN,[Official management guidance for forward-look...


In [26]:
for ticker, workflow in mag7_cio_results.items():
    debate = workflow.get("debate", {})
    responses = debate.get("debate_responses", [])
    revised_packets = debate.get("revised_packets", [])

    print("\n" + "=" * 100)
    print(f"{ticker} | debate_triggered={debate.get('triggered')} | conflict_level={debate.get('trigger', {}).get('conflict_level')}")
    print("=" * 100)

    for i, response in enumerate(responses):
        packet_agent = None
        if i < len(revised_packets):
            packet_agent = revised_packets[i].get("agent_name")

        agent_name = response.get("agent_name") or packet_agent or f"agent_{i}"
        response_type = response.get("response_type", "revision")

        print(f"\n--- {agent_name} | {response_type} ---")

        print("\nAgreement:")
        agreement = response.get("agreement")
        if isinstance(agreement, dict):
            for other_agent, text in agreement.items():
                print(f"- with {other_agent}: {text}")
        elif isinstance(agreement, list):
            for item in agreement:
                print(f"- {item}")
        else:
            print(agreement)

        print("\nDisagreement:")
        disagreement = response.get("disagreement")
        if isinstance(disagreement, dict):
            for other_agent, text in disagreement.items():
                print(f"- with {other_agent}: {text}")
        elif isinstance(disagreement, list):
            for item in disagreement:
                print(f"- {item}")
        else:
            print(disagreement)

        print("\nRevised view:")
        print(f"- short_term: {response.get('revised_short_term_stance')}")
        print(f"- long_term: {response.get('revised_long_term_stance')}")
        print(f"- confidence: {response.get('revised_confidence')}")
        print(f"- evidence_needed: {response.get('evidence_needed')}")



TSLA | debate_triggered=True | conflict_level=high

--- news_macro | CIO Committee Revision ---

Agreement:
- with market_technical: I agree with the market_technical agent that the price remaining below the 200-day moving average ($402.86) is a significant long-term bearish indicator. I also concur that the recent price rally is on weak volume, which suggests a lack of conviction and supports my view that the negative fundamentals will ultimately prevail over a short-term narrative-driven bounce.
- with fundamental: I strongly agree with the fundamental agent's key findings that Tesla is experiencing a 'significant quarter-over-quarter decline' in revenue and profitability, and that its 'extremely high valuation' is at risk due to this slowing growth. This aligns perfectly with the core of my bearish thesis.

Disagreement:
- with market_technical: I disagree with the market_technical agent's short-term 'Bullish' stance. While I acknowledge the bullish momentum indicators (MACD, RSI),